In [1]:
import sys
from pathlib import Path
repo_root = Path().resolve().parent  
sys.path.append(str(repo_root))

In [2]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from plotly import express as px
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score
from sklearn.model_selection import train_test_split, learning_curve, LearningCurveDisplay, RandomizedSearchCV
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.ensemble import RandomForestRegressor, AdaBoostRegressor, GradientBoostingRegressor, StackingRegressor
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import Ridge, ElasticNet,  ElasticNetCV, RidgeCV, LassoCV, SGDRegressor, LassoLarsCV, LinearRegression
import matplotlib.pyplot as plt
from src.data.Segment_Slicer import SegmentSlicer 
SegmentSlicer = SegmentSlicer()
from scipy.optimize import fsolve
from scipy.interpolate import interp1d
import pickle
from src.data.Theoretical_Best_Time_Estimator import TheoreticalBestTimeEstimator
TheoreticalBestTimeEstimator = TheoreticalBestTimeEstimator()

# 1. Load Data

## 1.1 Load segments

In [3]:
raw_df= pd.read_parquet(repo_root / 'data' / 'processed' / 'reunion_segments_cleaned.parquet')
df= raw_df.copy()

In [4]:
print(f"📊 Dataset: {len(raw_df)} segments")
print(f"   Running: {(raw_df['activity_type']=='Run').sum()}")
print(f"   Cycling: {(raw_df['activity_type']=='Ride').sum()}")

📊 Dataset: 6341 segments
   Running: 3481
   Cycling: 2860


In [ ]:
run_df = df[df['activity_type']=='Run'].copy()

In [ ]:
sections_dict_run = {}
for idx, row in run_df.iterrows():
    segment_id = row['segment_id']
    # Charger altitude_profile, distance_profile, coordinates
    sections = SegmentSlicer.cut_segment(row['altitude_profile'], row['distance_profile'], row['coordinates'])
    sections_dict_run[segment_id] = sections

## 1.3 Extract features

In [7]:
def compute_segment_time_fast(sections, segment_distance_km, lookup_dict):
    """
    Calcule le temps total RAPIDEMENT en utilisant la lookup table.
    
    Args:
        sections: Liste de dict avec 'distance' (m) et 'grade' (%)
        segment_distance_km: Distance totale du segment en km
        lookup_dict: Dictionnaire retourné par build_lookup_table_3d()
    
    Returns:
        float: Temps total en secondes
    """
    interpolator = lookup_dict['interpolator']
    
    total_time = 0.0
    
    for section in sections:
        sect_dist_km = section['distance'] / 1000  # convertir en km
        grade = section['grade']
        
        # Lookup dans la table 3D: (segment_dist, section_dist, grade) → time(s)
        time_seconds = interpolator([segment_distance_km, sect_dist_km, grade])[0]
        
        total_time += time_seconds
    
    return total_time

In [8]:
def extract_features_from_sections(sections, segment_id=None, df=None):
    """
    Extrait des features simples à partir des sections d'un segment.
    
    Input: sections (list of dict) - output de segment_slicer.cut_segment()
    Output: dict of features
    """
    
    if not sections or len(sections) == 0:
        return None
    
    # ========== Features Basiques ==========
    total_distance = sum(s['distance'] for s in sections) / 1000  # en km
    total_elevation_gain = sum(s['elevation_gain'] for s in sections)
    total_elevation_loss = sum(s['elevation_loss'] for s in sections)
    
    # Grades
    all_grades = [s['grade'] for s in sections]
    avg_grade = np.mean(all_grades)
    max_grade = max(s['max_grade'] for s in sections)
    min_grade = min(s['min_grade'] for s in sections)
    
    # ========== Features d'Ordre (Capture la Séquence) ==========
    
    # 1. Distribution des montées (early vs late)
    early_third_distance = total_distance * 1000 * 0.33
    late_third_distance = total_distance * 1000 * 0.67
    
    early_climb_gain = sum(s['elevation_gain'] for s in sections 
                           if s['start_distance'] < early_third_distance)
    late_climb_gain = sum(s['elevation_gain'] for s in sections 
                          if s['start_distance'] > late_third_distance)
    
    early_climb_ratio = early_climb_gain / (total_elevation_gain + 1e-6)
    late_climb_ratio = late_climb_gain / (total_elevation_gain + 1e-6)
    
    # 2. Grade pondéré par position (effet fatigue)
    weighted_grade = 0
    for i, s in enumerate(sections):
        position_weight = 1 + (i / len(sections)) * 0.5  # 1.0 à 1.5x
        weighted_grade += s['grade'] * position_weight * s['distance']
    weighted_grade /= (total_distance * 1000)
    
    # 3. Position de la section la plus dure
    hardest_idx = np.argmax([s['grade'] * s['distance'] for s in sections])
    hardest_section_position = hardest_idx / len(sections)  # 0 à 1
    
    # 4. Variabilité du terrain
    grade_variance = np.mean([s['grade_variance'] for s in sections])
    
    # 5. Stats par tiers du segment
    first_third = [s for s in sections if s['start_distance'] < early_third_distance]
    middle_third = [s for s in sections 
                    if early_third_distance <= s['start_distance'] <= late_third_distance]
    last_third = [s for s in sections if s['start_distance'] > late_third_distance]
    
    first_third_avg_grade = np.mean([s['grade'] for s in first_third]) if first_third else 0
    middle_third_avg_grade = np.mean([s['grade'] for s in middle_third]) if middle_third else 0
    last_third_avg_grade = np.mean([s['grade'] for s in last_third]) if last_third else 0
    
    # 6. Compter les types de sections
    n_climbs = sum(1 for s in sections if s['type'] in ['climb', 'uphill'])
    n_descents = sum(1 for s in sections if s['type'] in ['descent', 'downhill'])
    n_flats = sum(1 for s in sections if s['type'] == 'flat')

    # 7. Features lié au data leakage
    best_time = df.loc[df['segment_id'] == segment_id, 'best_time'].iloc[0]
    avg_top_10_time = df.loc[df['segment_id'] == segment_id, 'average_top_10_time'].iloc[0]
    total_effort_count = df.loc[df['segment_id'] == segment_id, 'total_effort_count'].iloc[0]
    inv_total_effort_count = 1 / (total_effort_count + 1e-6)  

    # 8. Distance par catégorie de montée
    cat_hc_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat HC') / 1000  # en km
    cat_1_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat 1') / 1000  # en km
    cat_2_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat 2') / 1000  # en km
    cat_3_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat 3') / 1000  # en km  
    cat_4_distance = sum(s['distance'] for s in sections if s['category'] == 'Cat 4') / 1000  # en km
    
    uphill_distance = sum(s['distance'] for s in sections if s['type'] == 'uphill') / 1000  # en km
    downhill_distance = sum(s['distance'] for s in sections if s['type'] == 'downhill') / 1000  # en km
    flat_distance = sum(s['distance'] for s in sections if s['type'] == 'flat') / 1000  # en km

    # 9. Score physique
    with open('../src/models/250W_2900Ws_05_power_profile_time_lookup_table.pkl', 'rb') as f:
        lookup_dict = pickle.load(f)
    time_score = compute_segment_time_fast(sections, total_distance, lookup_dict=lookup_dict)

    
        
    
    # ========== Assembler le dictionnaire de features ==========
    features = {
        # Basiques
        'total_distance_km': total_distance,
        'total_elevation_gain': total_elevation_gain,
        'total_elevation_loss': total_elevation_loss,
        #'avg_grade': avg_grade,
        'max_grade': max_grade,
        'min_grade': min_grade,
        'grade_variance': grade_variance,
        
        # Ordre et fatigue
        'early_climb_ratio': early_climb_ratio,
        'late_climb_ratio': late_climb_ratio,
        'weighted_grade': weighted_grade,
        'hardest_section_position': hardest_section_position,
        
        # Tiers
        'first_third_avg_grade': first_third_avg_grade,
        'middle_third_avg_grade': middle_third_avg_grade,
        'last_third_avg_grade': last_third_avg_grade,
        
        # Comptages
        #'n_sections': len(sections),
        #'n_climbs': n_climbs,
        #'n_descents': n_descents,
        #'n_flats': n_flats,

        # Data leakage
        #'best_time': best_time,
        #'avg_top_10_time': avg_top_10_time,
        #'total_effort_count': total_effort_count,
        #'inv_total_effort_count': inv_total_effort_count

        # Distance par catégorie de montée
        #'cat_hc_distance_km': cat_hc_distance,
        'cat_1_distance_km': cat_1_distance,
        'cat_2_distance_km': cat_2_distance,
        'cat_3_distance_km': cat_3_distance,
        'cat_4_distance_km': cat_4_distance,
        'uphill_distance_km': uphill_distance,
        'downhill_distance_km': downhill_distance,
        'flat_distance_km': flat_distance,

        # Polynomial features 
        'flat_cat_4_interaction': flat_distance * cat_4_distance,
        'flat_cat_3_interaction': flat_distance * cat_4_distance,
        'flat_cat_2_interaction': flat_distance * cat_4_distance,
        'flat_cat_1_interaction': flat_distance * cat_4_distance,
        'flat_DH_interaction': flat_distance * downhill_distance,
        'flat_UH_interaction': flat_distance * uphill_distance,
        'UH_cat_4_interaction': uphill_distance * cat_4_distance,
        'UH_cat_3_interaction': uphill_distance * cat_3_distance,
        'UH_cat_2_interaction': uphill_distance * cat_2_distance,
        'UH_cat_1_interaction': uphill_distance * cat_1_distance,
        'DH_cat_4_interaction': downhill_distance * cat_4_distance,
        'DH_cat_3_interaction': downhill_distance * cat_3_distance,
        'DH_cat_2_interaction': downhill_distance * cat_2_distance,
        'DH_cat_1_interaction': downhill_distance * cat_1_distance,
        
        # Physics scores
        'time_score': time_score,


    }
    
    return features

In [9]:
def extract_features_for_dataframe(df, sections_dict):
    features_list = []
    segment_ids = []  # ← AJOUTER
    
    for idx, row in df.iterrows():
        segment_id = row['segment_id']
        
        if segment_id not in sections_dict:
            print(f"Warning: No sections found for segment {segment_id}")
            continue
        
        sections = sections_dict[segment_id]
        features = extract_features_from_sections(sections, segment_id=segment_id, df=df)
        
        if features is not None:
            features_list.append(features)
            segment_ids.append(segment_id)  # ← AJOUTER
    
    features_df = pd.DataFrame(features_list, index=segment_ids)  # ← MODIFIER
    print(f"✓ Extracted features for {len(features_df)} / {len(df)} segments")
    
    return features_df

In [ ]:
features_run_df = extract_features_for_dataframe(run_df, sections_dict_ride)

✓ Extracted features for 2857 / 2860 segments


## 1.4 Cleaning

Cleaning : Too fast top 1 rider

In [ ]:
mask_too_fast_best_rider = (run_df['best_time'] < run_df['average_top_10_time'] / 2)
run_df.loc[mask_too_fast_best_rider, 'best_time'] = run_df.loc[mask_too_fast_best_rider, 'average_top_10_time'].astype(int)
print(f"🚫 Removed {(mask_too_fast_best_rider).sum()} outliers where best_time < average_top_10_time / 2")

🚫 Removed 62 outliers where best_time < average_top_10_time / 2


Cleaning : Too steep slope ( |slope|>25% )

In [ ]:
mask_too_steep_slope = (features_run_df['max_grade'] <= 30) & (features_run_df['min_grade'] >= -30)
print(f"Removing {len(features_run_df) - mask_too_steep_slope.sum()} outliers from training set based on grade thresholds.")
features_run_df = features_run_df.loc[mask_too_steep_slope]
sections_dict_ride = {k: v for k, v in sections_dict_ride.items() if k in run_df['segment_id'].values}

Removing 451 outliers from training set based on grade thresholds.


Cleaned databases:
- sections_dict_ride
- df_ride

## 1.2 Load medium confidence T1 segments 

In [93]:
T1_MC_active_learning_ride = pd.read_csv(repo_root / 'data' / 'processed' / 'T1_active_learning_threshold_5_ride.csv')
T1_road_naming_ride = pd.read_csv(repo_root / 'data' / 'processed' / 'T1_road_naming_ride.csv')
segments_manually_labeled = pd.read_csv(repo_root / 'data' / 'processed' / 'segments_manually_labeled.csv')
T1_segments_manually_labeled_ride = segments_manually_labeled[(segments_manually_labeled['technicality'] == 1) & (segments_manually_labeled['segment_id'].isin(df[df['activity_type'] == 'Ride']['segment_id']))].copy()
not_T1_segments_manually_labeled_ride = segments_manually_labeled[(segments_manually_labeled['technicality'] != 1) & (segments_manually_labeled['segment_id'].isin(df[df['activity_type'] == 'Ride']['segment_id']))].copy()

T1_MC_ride = pd.concat([T1_MC_active_learning_ride, T1_road_naming_ride, T1_segments_manually_labeled_ride]).drop_duplicates().reset_index(drop=True)
T1_MC_ride = T1_MC_ride.drop(T1_MC_ride[T1_MC_ride['segment_id'].isin(not_T1_segments_manually_labeled_ride['segment_id'])].index).reset_index(drop=True)
print(f"Total T1 Medium Confidence Ride segments labeled: {len(T1_MC_ride)}")

Total T1 Medium Confidence Ride segments labeled: 1348


In [ ]:
T1_MC_HE_ride = run_df[(run_df['total_effort_count']>5000) & (run_df['segment_id'].isin(T1_MC_ride['segment_id']))].copy()

## 1.3 Spliting  

In [ ]:
Train_ride, Test_ride = train_test_split(features_run_df, test_size=0.2, random_state=42)
Train_ride, Val_ride = train_test_split(Train_ride, test_size=0.25, random_state=42)
print(f"Train: {len(Train_ride)}, Val: {len(Val_ride)}, Test: {len(Test_ride)}")

Train: 1443, Val: 481, Test: 482


In [105]:
Train_T1_MC_ride = Train_ride[Train_ride.index.isin(T1_MC_ride['segment_id'])].copy()
Val_T1_MC_ride = Val_ride[Val_ride.index.isin(T1_MC_ride['segment_id'])].copy()
Test_T1_MC_ride = Test_ride[Test_ride.index.isin(T1_MC_ride['segment_id'])].copy()
print(f"Train T1 MC: {len(Train_T1_MC_ride)}, Val T1 MC: {len(Val_T1_MC_ride)}, Test T1 MC: {len(Test_T1_MC_ride)}")

Train T1 MC: 755, Val T1 MC: 261, Test T1 MC: 272


In [57]:
Train_T1_MC_HE_ride = Train_ride[Train_ride.index.isin(T1_MC_HE_ride['segment_id'])].copy()
Val_T1_MC_HE_ride = Val_ride[Val_ride.index.isin(T1_MC_HE_ride['segment_id'])].copy()
Test_T1_MC_HE_ride = Test_ride[Test_ride.index.isin(T1_MC_HE_ride['segment_id'])].copy()
print(f"Train T1 MC HE: {len(Train_T1_MC_HE_ride)}, Val T1 MC HE: {len(Val_T1_MC_HE_ride)}, Test T1 MC HE: {len(Test_T1_MC_HE_ride)}")

Train T1 MC HE: 315, Val T1 MC HE: 121, Test T1 MC HE: 122


# 2. T1 Best Time - Pipeline

## 2.1 X and Y

In [ ]:
"""
X_train_ride = Train_T1_MC_ride.copy()
y_train_ride = run_df.set_index('segment_id').loc[Train_T1_MC_ride.index, 'best_time']

X_test_ride = Test_T1_MC_ride.copy()
y_test_ride = run_df.set_index('segment_id').loc[Test_T1_MC_ride.index, 'best_time']

X_val_ride = Val_T1_MC_ride.copy()
y_val_ride = run_df.set_index('segment_id').loc[Val_T1_MC_ride.index, 'best_time']

"""
X_train_ride = Train_T1_MC_HE_ride.copy()
y_train_ride = run_df.set_index('segment_id').loc[Train_T1_MC_HE_ride.index, 'best_time']

X_test_ride = Test_T1_MC_HE_ride.copy()
y_test_ride = run_df.set_index('segment_id').loc[Test_T1_MC_HE_ride.index, 'best_time']

X_val_ride = Val_T1_MC_HE_ride.copy()
y_val_ride = run_df.set_index('segment_id').loc[Val_T1_MC_HE_ride.index, 'best_time']

## 2.2 Model

In [59]:
class ModelRouter4(BaseEstimator, RegressorMixin):
    def __init__(self, pipeline_1, pipeline_2, pipeline_3, pipeline_4,
                 threshold_1=1.0, threshold_2=2.0, threshold_3=5.0, 
                 segment_length_col='segment_length',
                 drop_routing_col=True):  # ← NOUVEAU paramètre
        self.pipeline_1 = pipeline_1
        self.pipeline_2 = pipeline_2
        self.pipeline_3 = pipeline_3
        self.pipeline_4 = pipeline_4
        self.threshold_1 = threshold_1
        self.threshold_2 = threshold_2
        self.threshold_3 = threshold_3
        self.segment_length_col = segment_length_col
        self.drop_routing_col = drop_routing_col  # ← NOUVEAU

    def _prepare_features(self, X):
        """Enlève la colonne de routing des features si demandé"""
        if self.drop_routing_col and self.segment_length_col in X.columns:
            return X.drop(columns=[self.segment_length_col])
        return X

    def fit(self, X, y):
        # Calculer les masques AVANT de retirer la colonne
        self.mask_1 = X[self.segment_length_col] <= self.threshold_1
        self.mask_2 = (X[self.segment_length_col] > self.threshold_1) & \
                      (X[self.segment_length_col] <= self.threshold_2)
        self.mask_3 = (X[self.segment_length_col] > self.threshold_2) & \
                      (X[self.segment_length_col] <= self.threshold_3)
        self.mask_4 = X[self.segment_length_col] > self.threshold_3

        # Préparer les features SANS la colonne de routing
        X_features = self._prepare_features(X)

        # Entraîner chaque pipeline sur son subset
        if np.any(self.mask_1):
            self.pipeline_1.fit(X_features[self.mask_1], y[self.mask_1])
        if np.any(self.mask_2):
            self.pipeline_2.fit(X_features[self.mask_2], y[self.mask_2])
        if np.any(self.mask_3):
            self.pipeline_3.fit(X_features[self.mask_3], y[self.mask_3])
        if np.any(self.mask_4):
            self.pipeline_4.fit(X_features[self.mask_4], y[self.mask_4])
        
        return self

    def predict(self, X):
        # Calculer les masques AVANT de retirer la colonne
        mask_1 = X[self.segment_length_col] <= self.threshold_1
        mask_2 = (X[self.segment_length_col] > self.threshold_1) & \
                 (X[self.segment_length_col] <= self.threshold_2)
        mask_3 = (X[self.segment_length_col] > self.threshold_2) & \
                 (X[self.segment_length_col] <= self.threshold_3)
        mask_4 = X[self.segment_length_col] > self.threshold_3

        # Préparer les features SANS la colonne de routing
        X_features = self._prepare_features(X)

        y_pred = np.zeros(len(X))
        if np.any(mask_1):
            y_pred[mask_1] = self.pipeline_1.predict(X_features[mask_1])
        if np.any(mask_2):
            y_pred[mask_2] = self.pipeline_2.predict(X_features[mask_2])
        if np.any(mask_3):
            y_pred[mask_3] = self.pipeline_3.predict(X_features[mask_3])
        if np.any(mask_4):
            y_pred[mask_4] = self.pipeline_4.predict(X_features[mask_4])
        
        return y_pred

In [ ]:
features_to_exclude = ['total_distance_km', 'total_elevation_gain', 'total_elevation_loss','min_grade','max_grade', 'hardest_section_position','grade_variance']
X_train_for_selection = X_train_ride.select_dtypes(include=[np.number]).drop(columns=features_to_exclude, errors='ignore')
sfs = SequentialFeatureSelector(LinearRegression(), n_features_to_select='auto', direction='forward', scoring='neg_root_mean_squared_error', tol=0.1)
sfs.fit(X_train_for_selection, y_train_ride)
selected_features_1011 = X_train_for_selection.columns[sfs.get_support()].tolist()
if 'total_distance_km' not in selected_features_1011:
    selected_features_1011.insert(0, 'total_distance_km')
.index
print("Selected features:", selected_features_1011)

Selected features: ['total_distance_km', 'weighted_grade', 'cat_2_distance_km', 'downhill_distance_km', 'flat_distance_km', 'time_score']


In [165]:
#linear regression statmodel
import statsmodels.api as sm
X_train_sm = sm.add_constant(X_train_ride[selected_features_1011].drop(columns='total_distance_km'))
model_sm = sm.OLS(y_train_ride, X_train_sm).fit()
print(model_sm.summary())

                            OLS Regression Results                            
Dep. Variable:              best_time   R-squared:                       0.974
Model:                            OLS   Adj. R-squared:                  0.974
Method:                 Least Squares   F-statistic:                     2351.
Date:                Mon, 15 Dec 2025   Prob (F-statistic):          1.85e-243
Time:                        16:47:49   Log-Likelihood:                -1615.0
No. Observations:                 315   AIC:                             3242.
Df Residuals:                     309   BIC:                             3265.
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                           coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------------------------------------------------------------
const                   13.6753 

In [166]:
pipeline_1 = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RidgeCV(cv=5))
    ])  

pipeline_2 = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RidgeCV(cv=5))
    ])
pipeline_3 = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RidgeCV(cv=5))
    ])  
pipeline_4 = Pipeline([
    ('scaler', StandardScaler()),
    ('model', RidgeCV(cv=5))
    ])  

model_router = ModelRouter4(pipeline_1, pipeline_2, pipeline_3, pipeline_4, threshold_1=1.0, threshold_2=2.0,threshold_3=4.0, segment_length_col='total_distance_km', drop_routing_col=True)
model_router.fit(X_train_ride, y_train_ride)

,pipeline_1,Pipeline(step...dgeCV(cv=5))])
,pipeline_2,Pipeline(step...dgeCV(cv=5))])
,pipeline_3,Pipeline(step...dgeCV(cv=5))])
,pipeline_4,Pipeline(step...dgeCV(cv=5))])
,threshold_1,1.0
,threshold_2,2.0
,threshold_3,4.0
,segment_length_col,'total_distance_km'
,drop_routing_col,True
,copy,True
,with_mean,True


In [112]:
def plotting_MAE_bins(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred):
    # Définir les bins de durée
    bins = [0, 60, 300, 900, 3600, np.inf]
    labels = ['<1min', '1-5min', '5-15min', '15-60min', '>60min']

    # Créer DataFrames temporaires avec bins
    train_df = pd.DataFrame({
        'actual': y_train,
        'pred': y_train_pred
    })
    train_df['bin'] = pd.cut(train_df['actual'], bins=bins, labels=labels)

    val_df = pd.DataFrame({
        'actual': y_val,
        'pred': y_val_pred
    })
    val_df['bin'] = pd.cut(val_df['actual'], bins=bins, labels=labels)

    test_df = pd.DataFrame({
        'actual': y_test,
        'pred': y_test_pred
    })
    test_df['bin'] = pd.cut(test_df['actual'], bins=bins, labels=labels)
    # Calculer les métriques par bin
    results = []
    for bin_label in labels:
        train_mask = train_df['bin'] == bin_label
        val_mask = val_df['bin'] == bin_label
        test_mask = test_df['bin'] == bin_label
        
        if train_mask.sum() > 0 and val_mask.sum() > 0 and test_mask.sum() > 0:
            train_mae = np.mean(np.abs(train_df.loc[train_mask, 'actual'] - train_df.loc[train_mask, 'pred']))
            val_mae = np.mean(np.abs(val_df.loc[val_mask, 'actual'] - val_df.loc[val_mask, 'pred']))
            test_mae = np.mean(np.abs(test_df.loc[test_mask, 'actual'] - test_df.loc[test_mask, 'pred']))
            
            train_n = train_mask.sum()
            val_n = val_mask.sum()
            test_n = test_mask.sum()
            
            results.append({
                'Durée': bin_label,
                'Train N': train_n,
                'Train MAE': f"{train_mae:.0f}s",
                'Val N': val_n,
                'Val MAE': f"{val_mae:.0f}s",
                'Test N': test_n,
                'Test MAE': f"{test_mae:.0f}s"
            })

    # Créer et afficher le tableau
    results_df = pd.DataFrame(results)

    print("\n" + "="*70)
    print("PERFORMANCE PAR BIN DE DURÉE")
    print("="*70)
    print(results_df.to_string(index=False))
    print("="*70)

    # Métriques globales
    print(f"\nGLOBAL - Train RMSE: {root_mean_squared_error(y_train, y_train_pred):.2f}")
    print(f"GLOBAL - Train MAE: {mean_absolute_error(y_train, y_train_pred):.1f}s")
    print(f"GLOBAL - Validation RMSE: {root_mean_squared_error(y_val, y_val_pred):.2f}")
    print(f"GLOBAL - Validation MAE: {mean_absolute_error(y_val, y_val_pred):.1f}s")
    return None

In [113]:
def plotting_pred_vs_actual_LC(pipeline, X_train, y_train, X_val, y_val, X_test, y_test):
    
    pipeline.fit(X_train.select_dtypes(include=[np.number]), y_train)
    y_val_pred = pipeline.predict(X_val.select_dtypes(include=[np.number]))
    y_train_pred = pipeline.predict(X_train.select_dtypes(include=[np.number]))
    y_test_pred = pipeline.predict(X_test.select_dtypes(include=[np.number]))

    plotting_MAE_bins(y_train, y_train_pred, y_val, y_val_pred, y_test, y_test_pred)

    train_size_abs, train_scores, test_scores = learning_curve(
        pipeline, X_train.select_dtypes(include=[np.number]), y_train,
        cv=5, scoring='neg_root_mean_squared_error', train_sizes=np.linspace(0.1, 1.0, 10), n_jobs=-1
    )
    fig = make_subplots(rows=1, cols=4, subplot_titles=("Learning Curve", "Train Prediction vs Actual", "Validation Prediction vs Actual", "Test Prediction vs Actual"))
    
    # Plot learning curve
    train_sizes = np.linspace(0.1, 1.0, 10) * len(X_train)
    train_scores_mean = -np.mean(train_scores, axis=1)
    test_scores_mean = -np.mean(test_scores, axis=1)
    fig.add_trace(
        go.Scatter(
            x=train_sizes,
            y=train_scores_mean,
            mode='lines+markers',
            name='Train Score',
            line=dict(color='blue'),
            hovertemplate='Size: %{x:.0f}<br>MAE: %{y:.0f}s<extra></extra>'
        ),
        row=1, col=1
    )
    fig.add_trace(
        go.Scatter(
            x=train_sizes,
            y=test_scores_mean,
            mode='lines+markers',
            name='Val Score',
            line=dict(color='red'),
            hovertemplate='Size: %{x:.0f}<br>MAE: %{y:.0f}s<extra></extra>'
        ),
        row=1, col=1
    )
    fig.update_xaxes(title_text="Training Examples", row=1, col=1)
    fig.update_yaxes(title_text="MAE", row=1, col=1, range=[min(train_scores_mean)*0.9, min(test_scores_mean)*1.3])
    
    ###################### PLOT Train Prediction vs Actual ######################
    # IMPORTANT: Reset index de Train_df pour aligner avec X_train

    
    # Créer DataFrame avec les prédictions et les segment_ids alignés par position
    X_train_with_info = pd.DataFrame({
        'segment_id': X_train.index,
        'actual': y_train.values if isinstance(y_train, pd.Series) else y_train,
        'predicted': y_train_pred
    })
    
    # Plot prediction vs actual values
    fig.add_trace(
        go.Scatter(
            x=X_train_with_info['actual'],
            y=X_train_with_info['predicted'],
            mode='markers',  
            marker=dict(symbol="x-thin", size=5, line=dict(width=1, color='red')),
            name='Train Predictions',
            text=X_train_with_info['segment_id'],
            customdata=X_train_with_info['segment_id'],
            hovertemplate="<b>Segment ID:</b> %{text}<br>" +
                        "<b>Actual:</b> %{x:.0f}s<br>" +
                        "<b>Predicted:</b> %{y:.0f}s<extra></extra>"
        ),
        row=1, col=2
    )
    # Add a line for perfect predictions
    fig.add_trace(
        go.Scatter(
            x=[10, 10000],
            y=[10, 10000],
            mode='lines',
            line=dict(dash='dash', color='black'),
            showlegend=False
        ),
        row=1, col=2
    )
    fig.update_xaxes(title_text="Actual", row=1, col=2, type='log', range=[1,4])
    fig.update_yaxes(title_text="Predicted", row=1, col=2, type='log', range=[1,4])
    
    ###################### PLOT Validation Prediction vs Actual ######################
    
    # Créer DataFrame avec les prédictions et les segment_ids alignés par position
    X_val_with_info = pd.DataFrame({
        'segment_id': X_val.index,
        'actual': y_val.values if isinstance(y_val, pd.Series) else y_val,
        'predicted': y_val_pred
    })
    
    # Plot prediction vs actual values
    fig.add_trace(
        go.Scatter(
            x=X_val_with_info['actual'],
            y=X_val_with_info['predicted'],
            mode='markers',  
            marker=dict(symbol="x-thin", size=5, line=dict(width=1, color='green')),
            name='Validation Predictions',
            text=X_val_with_info['segment_id'],
            customdata=X_val_with_info['segment_id'],
            hovertemplate="<b>Segment ID:</b> %{text}<br>" +
                        "<b>Actual:</b> %{x:.0f}s<br>" +
                        "<b>Predicted:</b> %{y:.0f}s<extra></extra>"
        ),
        row=1, col=3
    )
    # Add a line for perfect predictions
    fig.add_trace(
        go.Scatter(
            x=[10, 10000],
            y=[10, 10000],
            mode='lines',
            line=dict(dash='dash', color='black'),
            showlegend=False
        ),
        row=1, col=3
    )
    fig.update_xaxes(title_text="Actual", row=1, col=3, type='log', range=[1,4])
    fig.update_yaxes(title_text="Predicted", row=1, col=3, type='log', range=[1,4])
   
    ###################### PLOT Test Prediction vs Actual ######################
    
    # Créer DataFrame avec les prédictions et les segment_ids alignés par position
    X_test_with_info = pd.DataFrame({
        'segment_id': X_test.index,
        'actual': y_test.values if isinstance(y_test, pd.Series) else y_test,
        'predicted': y_test_pred
    })
    
    # Plot prediction vs actual values
    fig.add_trace(
        go.Scatter(
            x=X_test_with_info['actual'],
            y=X_test_with_info['predicted'],
            mode='markers',  
            marker=dict(symbol="x-thin", size=5, line=dict(width=1, color='red')),
            name='Test Predictions',
            text=X_test_with_info['segment_id'],
            customdata=X_test_with_info['segment_id'],
            hovertemplate="<b>Segment ID:</b> %{text}<br>" +
                        "<b>Actual:</b> %{x:.0f}s<br>" +
                        "<b>Predicted:</b> %{y:.0f}s<extra></extra>"
        ),
        row=1, col=4
    )
    # Add a line for perfect predictions
    fig.add_trace(
        go.Scatter(
            x=[10, 10000],
            y=[10, 10000],
            mode='lines',
            line=dict(dash='dash', color='black'),
            showlegend=False
        ),
        row=1, col=4
    )
    fig.update_xaxes(title_text="Actual", row=1, col=4, type='log', range=[1,4])
    fig.update_yaxes(title_text="Predicted", row=1, col=4, type='log', range=[1,4])

    # Update layout
    fig.update_layout(
        height=500,
        width=1400,
        title_text=f"Model Evaluation",
        hovermode='closest'
    )
    fig.show()
    return None

In [167]:
plotting_pred_vs_actual_LC(model_router, X_train_ride, y_train_ride, X_val_ride, y_val_ride, X_test_ride, y_test_ride)


PERFORMANCE PAR BIN DE DURÉE
   Durée  Train N Train MAE  Val N Val MAE  Test N Test MAE
   <1min       75        6s     31      9s      29       7s
  1-5min      170       20s     64     26s      66      27s
 5-15min       63       50s     23     61s      24      56s
15-60min        7       72s      3    182s       3     152s

GLOBAL - Train RMSE: 36.88
GLOBAL - Train MAE: 23.8s
GLOBAL - Validation RMSE: 54.54
GLOBAL - Validation MAE: 32.4s


c:\Users\coren\miniconda3\envs\tf\lib\site-packages\sklearn\model_selection\_validation.py:516: FitFailedWarning:


5 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\coren\miniconda3\envs\tf\lib\site-packages\sklearn\model_selection\_validation.py", line 859, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "C:\Users\coren\AppData\Local\Temp\ipykernel_23832\2657000260.py", line 42, in fit
  File "c:\Users\coren\miniconda3\envs\tf\lib\site-packages\sklearn\base.py", line 1365, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "c:\Users\coren\miniconda3\envs\tf\lib\site-pa

# 3. Theorical Best Time - Pipeline

In [119]:
y_test_pred = model_router.predict(X_test_ride)

In [ ]:
Test_df = pd.DataFrame({
    'segment_id': X_test_ride.index,
    'best_time': y_test_ride.values,
    'T1_time': y_test_pred,
    'total_effort_count': run_df.set_index('segment_id').loc[X_test_ride.index, 'total_effort_count']
})

In [121]:
Test_df_new = TheoreticalBestTimeEstimator.estimated_th_best_time(Test_df)
Test_df["theoretical_best_time"] = Test_df_new["theoretical_best_time"]

In [122]:
Test_df['TH_time_T1_time_ratio'] = Test_df['theoretical_best_time'] / Test_df['T1_time']
Test_df['best_time_T1_time_ratio'] = Test_df['best_time'] / Test_df['T1_time']

# 4. Classifier

## 4.1 On the whole dataset

In [ ]:
X_whole_ride = features_run_df.copy()
y_whole_ride = run_df.set_index('segment_id').loc[X_whole_ride.index, 'best_time']
y_whole_pred = model_router.predict(X_whole_ride)

In [ ]:
Whole_df = pd.DataFrame({
    'segment_id': X_whole_ride.index,
    'best_time': y_whole_ride.values,
    'T1_time': y_whole_pred,
    'total_effort_count': run_df.set_index('segment_id').loc[X_whole_ride.index, 'total_effort_count'],
    'tenth_best_time': run_df.set_index('segment_id').loc[X_whole_ride.index, 'tenth_best_time'],
    'average_top_10_time': run_df.set_index('segment_id').loc[X_whole_ride.index, 'average_top_10_time']
})

In [143]:
Whole_df_new = TheoreticalBestTimeEstimator.estimated_th_best_time(Whole_df)
Whole_df["theoretical_best_time"] = Whole_df_new["theoretical_best_time"]

In [144]:
Whole_df['TH_time_T1_time_ratio'] = Whole_df['theoretical_best_time'] / Whole_df['T1_time']
Whole_df['best_time_T1_time_ratio'] = Whole_df['best_time'] / Whole_df['T1_time']
Whole_df['tenth_best_time_T1_time_ratio'] = Whole_df['tenth_best_time'] / Whole_df['T1_time']
Whole_df['average_top_10_time_T1_time_ratio'] = Whole_df['average_top_10_time'] / Whole_df['T1_time']

In [153]:
from sklearn.cluster import KMeans
ratio = 'average_top_10_time_T1_time_ratio'
X_cluster_whole = Whole_df[[ratio]].copy()
kmeans = KMeans(n_clusters=5, random_state=42)
kmeans.fit(X_cluster_whole)
Whole_df['cluster'] = kmeans.labels_
centers_whole = kmeans.cluster_centers_
labeled_ids = segments_manually_labeled['segment_id'].values
Whole_df_labeled = Whole_df[Whole_df['segment_id'].isin(labeled_ids)].copy()
tech_map = segments_manually_labeled.set_index('segment_id')['technicality']
Whole_df_labeled['technicality'] = Whole_df_labeled['segment_id'].map(tech_map)

In [154]:
# plot technicality vs besti
fig = go.Figure()
for tech in sorted(Whole_df_labeled['technicality'].unique()):
    tech_data = Whole_df_labeled[Whole_df_labeled['technicality'] == tech]
    fig.add_trace(go.Scatter(
        x=tech_data[ratio],
        y=tech_data['technicality'],
        mode='markers',
        name=f'Tech {int(tech)}',
        marker=dict(size=8),
        hovertemplate=
            "<b>Segment ID:</b> %{customdata[0]}<br>" +
            "<b>Technicality:</b> %{y}<br>" +
            "<b>Best Time / T1 Time Ratio:</b> %{x:.2f}<br>" +
            "<b>Best Time:</b> %{customdata[1]:.0f}s<br>" +
            "<b>T1 Time:</b> %{customdata[2]:.0f}s<br>" +
            "<extra></extra>",
        customdata=np.stack((tech_data['segment_id'], tech_data['best_time'], tech_data['T1_time']), axis=-1)
    ))
fig.update_layout(
    title=f"Labeled Segments: Technicality vs {ratio}",
    xaxis_title=f"{ratio}",
    yaxis_title="Technicality Level",
    yaxis=dict(tickmode='linear', tick0=1, dtick=1),
    hovermode='closest'
)
fig.show()


In [ ]:
thresholds = [1.50, 2, 2.6, 4]  

def classify_by_ratio(ratio):
    if ratio < thresholds[0]:
        return 1
    elif ratio < thresholds[1]:
        return 2
    elif ratio < thresholds[2]:
        return 3
    elif ratio < thresholds[3]:
        return 4
    else:
        return 5

Whole_df_labeled['manual_cluster'] = Whole_df_labeled[ratio].apply(classify_by_ratio)

In [156]:
## plot confusion matrix of whole ride clustering using labeled segments in segments_manually_labeled
from sklearn.metrics import confusion_matrix
conf_matrix_whole = confusion_matrix(Whole_df_labeled['technicality'], Whole_df_labeled['manual_cluster'])
import plotly.graph_objects as go
fig = go.Figure(data=go.Heatmap(
    z=conf_matrix_whole,
    x=[f'T{i}' for i in range(1,6)],
    y=[f'T{i}' for i in range(1,6)],
    colorscale='Blues',
    text=conf_matrix_whole,
    texttemplate='%{text}',
    textfont={"size": 14}
))
fig.update_layout(
    title='Confusion Matrix (Whole Ride): Technicality (rows) vs Clusters (cols)',
    xaxis_title='Predicted Cluster',
    yaxis_title='True Technicality',
    height=500
)
fig.show()

In [39]:
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import numpy as np
import pandas as pd
from sklearn.metrics import confusion_matrix, adjusted_rand_score, normalized_mutual_info_score
import seaborn as sns
import matplotlib.pyplot as plt

# ============================================================================
# 1. DATA PREPARATION
# ============================================================================

# Merge labeled data
labeled_ids = segments_manually_labeled['segment_id'].values
Whole_df_labeled = Whole_df[Whole_df['segment_id'].isin(labeled_ids)].copy()
tech_map = segments_manually_labeled.set_index('segment_id')['technicality']
Whole_df_labeled['technicality'] = Whole_df_labeled['segment_id'].map(tech_map)

# ============================================================================
# 2. DISTRIBUTION ANALYSIS
# ============================================================================

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=(
        'Ratio Distribution by Technicality (Manual Labels)',
        'Ratio Distribution by Cluster (K-Means)',
        'Ratio Boxplot by Technicality',
        'Ratio Boxplot by Cluster'
    ),
    specs=[[{'type': 'box'}, {'type': 'box'}],
           [{'type': 'box'}, {'type': 'box'}]]
)

colors = px.colors.qualitative.Plotly

# Plot 1: Distribution by Technicality
for tech in sorted(Whole_df_labeled['technicality'].unique()):
    data = Whole_df_labeled[Whole_df_labeled['technicality'] == tech]['TH_time_T1_time_ratio']
    fig.add_trace(
        go.Box(y=data, name=f'Tech {int(tech)}', marker_color=colors[int(tech)-1]),
        row=1, col=1
    )

# Plot 2: Distribution by Cluster
for cluster in sorted(Whole_df_labeled['cluster'].unique()):
    data = Whole_df_labeled[Whole_df_labeled['cluster'] == cluster]['TH_time_T1_time_ratio']
    fig.add_trace(
        go.Box(y=data, name=f'Cluster {cluster}', marker_color=colors[cluster]),
        row=1, col=2
    )

# Plot 3: Violin plot by Technicality
for tech in sorted(Whole_df_labeled['technicality'].unique()):
    data = Whole_df_labeled[Whole_df_labeled['technicality'] == tech]['TH_time_T1_time_ratio']
    fig.add_trace(
        go.Violin(y=data, name=f'Tech {int(tech)}', marker_color=colors[int(tech)-1]),
        row=2, col=1
    )

# Plot 4: Violin plot by Cluster
for cluster in sorted(Whole_df_labeled['cluster'].unique()):
    data = Whole_df_labeled[Whole_df_labeled['cluster'] == cluster]['TH_time_T1_time_ratio']
    fig.add_trace(
        go.Violin(y=data, name=f'Cluster {cluster}', marker_color=colors[cluster]),
        row=2, col=2
    )

fig.update_layout(height=800, showlegend=False, title_text="Distribution Analysis: Manual Labels vs Clusters")
fig.show()

# ============================================================================
# 3. CONFUSION MATRIX & MAPPING
# ============================================================================

# Create confusion matrix
conf_matrix = confusion_matrix(Whole_df_labeled['technicality'], Whole_df_labeled['cluster'])

# Find best mapping from clusters to technicality levels
from scipy.optimize import linear_sum_assignment
row_ind, col_ind = linear_sum_assignment(-conf_matrix)
cluster_to_tech = {col_ind[i]: row_ind[i] + 1 for i in range(len(col_ind))}

print("Best Cluster to Technicality Mapping:")
for cluster, tech in sorted(cluster_to_tech.items()):
    print(f"  Cluster {cluster} → Technicality {tech}")

# Apply mapping
Whole_df_labeled['predicted_technicality'] = Whole_df_labeled['cluster'].map(cluster_to_tech)

# Plot confusion matrix
fig = go.Figure(data=go.Heatmap(
    z=conf_matrix,
    x=[f'Cluster {i}' for i in range(5)],
    y=[f'Tech {i}' for i in range(1, 6)],
    colorscale='Blues',
    text=conf_matrix,
    texttemplate='%{text}',
    textfont={"size": 14}
))

fig.update_layout(
    title='Confusion Matrix: Technicality (rows) vs Clusters (cols)',
    xaxis_title='Predicted Cluster',
    yaxis_title='True Technicality',
    height=500
)
fig.show()

# ============================================================================
# 4. PERFORMANCE METRICS
# ============================================================================

# Calculate metrics
ari = adjusted_rand_score(Whole_df_labeled['technicality'], Whole_df_labeled['cluster'])
nmi = normalized_mutual_info_score(Whole_df_labeled['technicality'], Whole_df_labeled['cluster'])

# Accuracy after optimal mapping
accuracy = (Whole_df_labeled['technicality'] == Whole_df_labeled['predicted_technicality']).mean()

print("\n" + "="*60)
print("CLUSTERING PERFORMANCE METRICS")
print("="*60)
print(f"Adjusted Rand Index (ARI): {ari:.3f}")
print(f"  → Measures similarity between clusterings (1=perfect, 0=random)")
print(f"\nNormalized Mutual Information (NMI): {nmi:.3f}")
print(f"  → Measures information shared between clusterings (1=perfect)")
print(f"\nAccuracy (after optimal mapping): {accuracy:.1%}")
print(f"  → Percentage of correctly classified segments")
print("="*60)

# ============================================================================
# 5. DETAILED STATISTICS BY TECHNICALITY & CLUSTER
# ============================================================================

print("\n" + "="*60)
print("RATIO STATISTICS BY TECHNICALITY LEVEL")
print("="*60)
tech_stats = Whole_df_labeled.groupby('technicality')['TH_time_T1_time_ratio'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(3)
print(tech_stats)

print("\n" + "="*60)
print("RATIO STATISTICS BY CLUSTER")
print("="*60)
cluster_stats = Whole_df_labeled.groupby('cluster')['TH_time_T1_time_ratio'].agg([
    'count', 'mean', 'median', 'std', 'min', 'max'
]).round(3)
print(cluster_stats)

# ============================================================================
# 6. SCATTER PLOT WITH ERRORS HIGHLIGHTED
# ============================================================================

fig = go.Figure()

# Plot correctly classified points
correct = Whole_df_labeled[Whole_df_labeled['technicality'] == Whole_df_labeled['predicted_technicality']]
incorrect = Whole_df_labeled[Whole_df_labeled['technicality'] != Whole_df_labeled['predicted_technicality']]

for tech in sorted(Whole_df_labeled['technicality'].unique()):
    correct_tech = correct[correct['technicality'] == tech]
    fig.add_trace(go.Scatter(
        x=correct_tech['TH_time_T1_time_ratio'],
        y=[tech] * len(correct_tech),
        mode='markers',
        name=f'Tech {int(tech)} (correct)',
        marker=dict(size=10, color=colors[int(tech)-1], symbol='circle'),
        hovertemplate=
            "<b>Segment:</b> %{customdata[0]}<br>" +
            "<b>True Tech:</b> " + str(int(tech)) + "<br>" +
            "<b>Cluster:</b> %{customdata[1]}<br>" +
            "<b>Ratio:</b> %{x:.3f}<extra></extra>",
        customdata=np.stack((correct_tech['segment_id'], correct_tech['cluster']), axis=-1)
    ))

# Highlight misclassified points
fig.add_trace(go.Scatter(
    x=incorrect['TH_time_T1_time_ratio'],
    y=incorrect['technicality'],
    mode='markers',
    name='Misclassified',
    marker=dict(
        size=14, 
        color='red', 
        symbol='x',
        line=dict(width=2, color='darkred')
    ),
    hovertemplate=
        "<b>Segment:</b> %{customdata[0]}<br>" +
        "<b>True Tech:</b> %{y}<br>" +
        "<b>Predicted Tech:</b> %{customdata[1]}<br>" +
        "<b>Cluster:</b> %{customdata[2]}<br>" +
        "<b>Ratio:</b> %{x:.3f}<extra></extra>",
    customdata=np.stack((
        incorrect['segment_id'], 
        incorrect['predicted_technicality'],
        incorrect['cluster']
    ), axis=-1)
))

fig.update_layout(
    title=f'Technicality vs TH/T1 Ratio (Accuracy: {accuracy:.1%})',
    xaxis_title='TH_time / T1_time Ratio',
    yaxis_title='Technicality Level',
    yaxis=dict(tickmode='linear', tick0=1, dtick=1),
    height=500,
    hovermode='closest'
)
fig.show()

# ============================================================================
# 7. ANALYZE MISCLASSIFICATIONS
# ============================================================================

if len(incorrect) > 0:
    print("\n" + "="*60)
    print(f"MISCLASSIFIED SEGMENTS ({len(incorrect)} total)")
    print("="*60)
    print(incorrect[['segment_id', 'technicality', 'predicted_technicality', 
                     'cluster', 'TH_time_T1_time_ratio']].sort_values('technicality'))

Best Cluster to Technicality Mapping:
  Cluster 0 → Technicality 2
  Cluster 1 → Technicality 1
  Cluster 2 → Technicality 3
  Cluster 3 → Technicality 4
  Cluster 4 → Technicality 5



CLUSTERING PERFORMANCE METRICS
Adjusted Rand Index (ARI): 0.670
  → Measures similarity between clusterings (1=perfect, 0=random)

Normalized Mutual Information (NMI): 0.503
  → Measures information shared between clusterings (1=perfect)

Accuracy (after optimal mapping): 6.8%
  → Percentage of correctly classified segments

RATIO STATISTICS BY TECHNICALITY LEVEL
              count   mean  median    std    min    max
technicality                                           
1                37  0.950   0.919  0.424 -0.551  2.487
2                 3  1.575   1.364  0.734  0.968  2.391
3                 3  2.267   2.264  0.448  1.821  2.716
4                 1  3.508   3.508    NaN  3.508  3.508

RATIO STATISTICS BY CLUSTER
         count   mean  median    std    min    max
cluster                                           
0           38  0.921   0.920  0.339 -0.551  1.690
3            6  2.531   2.439  0.563  1.821  3.508



MISCLASSIFIED SEGMENTS (41 total)
          segment_id  technicality  predicted_technicality  cluster  \
12389853    12389853             1                       2        0   
27179013    27179013             1                       2        0   
18850640    18850640             1                       2        0   
26796802    26796802             1                       2        0   
21898469    21898469             1                       2        0   
14062595    14062595             1                       2        0   
26052767    26052767             1                       2        0   
22708090    22708090             1                       2        0   
22847565    22847565             1                       2        0   
6405426      6405426             1                       2        0   
17073484    17073484             1                       2        0   
10729105    10729105             1                       2        0   
13728085    13728085             1        